# Evaluating and Optimizing Prompts with DSPy

## Why not just write a prompt?

```python
# The raw API approach:
response = bedrock.invoke_model(
    modelId="anthropic.claude-haiku...",
    body=json.dumps({"messages": [{"role": "user", "content": "What is the population of Seattle, WA?"}]})
)
# Works... but what happens when you need to improve it?
# You edit the prompt string. Test manually. Hope for the best.
```

```python
# The DSPy approach:
class CityQA(dspy.Signature):
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()
# Same output today. The difference: this one is optimizable.
```

They produce the same output today, but the difference shows up when we need to optimize.

### What you'll build

A **city Q&A system** that answers population and population-density questions about 300 US cities, scored against ground-truth data.

- **Part 1: The Naive Approach** — Point an LLM at questions, see what happens
- **Part 2: Measure, Then Optimize** — Measure quality, optimize automatically, inspect and save
- **Part 3: Go Deeper** — Add reasoning, compose modules, upgrade the metric

---
# Part 1: The Naive Approach

You point an LLM at city questions with zero prompt engineering. Some answers are perfect. Others are wildly wrong. And you have no systematic way to know which is which.

## 1. What do we need?

DSPy handles LLM orchestration and optimization. We also need `pandas` to load the city data.

In [ ]:
!pip install "dspy>=3.1,<4" boto3 pandas --quiet

## 2. Can we talk to the model?

DSPy supports Bedrock natively via LiteLLM. We configure a single LM that all modules will use. DSPy works with any Bedrock model so you can swap the model ID to try Nova, Claude, or Llama.

In [ ]:
import dspy
print(f"DSPy {dspy.__version__}")


lm = dspy.LM("bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0")
dspy.configure(lm=lm)

# Verify the connection works
try:
    response = lm("Say 'hello' and nothing else.")
    print(f"Connection OK: {response}")
except Exception as e:
    print(f"\u274c Bedrock connection failed: {e}")
    print("\nTroubleshooting:")
    print("  1. Verify model access is enabled in the Bedrock console")
    print("  2. Check your AWS credentials (run 'aws sts get-caller-identity')")
    print("  3. Ensure your IAM role has 'bedrock-runtime:InvokeModel' permission")
    raise

## 3. What does our data look like?

Load `city_pop.csv`, clean up footnote markers in city names, parse population values, and compute population density.

In [ ]:
import pandas as pd
import re

df = pd.read_csv("city_pop.csv", encoding="utf-8-sig")

# Clean city names — strip footnote markers like [c], [d]
df["city"] = df["city"].apply(lambda x: re.sub(r"\[.*?\]", "", x).strip())

# Parse numeric columns
df["population"] = df["population"].apply(lambda x: int(str(x).replace(",", "")))
df["land_area_mi2"] = pd.to_numeric(df["land_area_mi2"].str.replace(",", ""), errors="coerce")

# Calculate density (people per square mile)
df["density"] = (df["population"] / df["land_area_mi2"]).round(1)

print(f"{len(df)} cities loaded")
df[["city", "state", "population", "density"]].head(5)


## 4. How do we turn data into questions?

Generate question/answer pairs from the dataframe, one population question and one density question per city. Split into training (first 20 cities → 40 examples) and test (next 10 cities → 20 examples).

In [ ]:
def make_examples(df_slice):
    examples = []
    for _, row in df_slice.iterrows():
        examples.append(dspy.Example(
            question=f"What is the population of {row['city']}, {row['state']}?",
            answer=str(row["population"])
        ).with_inputs("question"))
        examples.append(dspy.Example(
            question=f"What is the population density (people per square mile) of {row['city']}, {row['state']}?",
            answer=str(row["density"])
        ).with_inputs("question"))
    return examples

trainset = make_examples(df.iloc[:20])
testset = make_examples(df.iloc[20:30])

print(f"Training examples: {len(trainset)}")
print(f"Test examples: {len(testset)}")
print(f"\nSample: {trainset[0].question} → {trainset[0].answer}")

## 5. What goes in, what comes out?

You stop editing prompt strings. Instead, you declare what goes in and what comes out — DSPy calls this a **Signature**. The docstring becomes the instruction and the field descriptions guide the model's output format.

In [ ]:
class CityQA(dspy.Signature):
    """Answer factual questions about US cities using your knowledge."""
    question: str = dspy.InputField(desc="A question about a US city's population or demographics")
    answer: str = dspy.OutputField(desc="A precise numeric answer to the question")

## 6. What happens with zero prompt engineering?

Let's see what the model does with zero help. No examples, no tricks, no hand-tuned prompt. `dspy.Predict` runs the signature as-is. This is our baseline: the score to beat.

In [ ]:
baseline = dspy.Predict(CityQA)

def extract_number(text):
    """Extract the first number from text, handling commas and decimals."""
    text = str(text).replace(",", "")
    matches = re.findall(r"[\d]+\.?[\d]*", text)
    return float(matches[0]) if matches else None


# Run on several test examples to see the variance
for i, test_example in enumerate(testset[:3]):
    result = baseline(question=test_example.question)
    expected = extract_number(test_example.answer)
    predicted = extract_number(result.answer)
    pct_err = abs(predicted - expected) / expected * 100 if predicted and expected else float('inf')
    marker = "\u2705" if pct_err <= 5 else ("\u26a0\ufe0f" if pct_err <= 10 else "\u274c")
    print(f"--- Example {i+1} {marker} ---")
    print(f"  Question:  {test_example.question}")
    print(f"  Predicted: {result.answer}")
    print(f"  Expected:  {test_example.answer}")
    print(f"  Error:     {pct_err:.1f}%")
    print()

### What did DSPy actually send to the model?

Let's peek behind the curtain.

In [ ]:
dspy.inspect_history(n=1)

---
# Part 2: Measure, Then Optimize

You've seen the model get it wrong. But how wrong, and how often? And what if you could fix it automatically?

## 7. How wrong is the model, and how often?

We need a function that answers: *how wrong?* Extract the number from the model's response, compute % error against ground truth. Within 5% → perfect. Within 10% → acceptable. Anything else → fail.

💡 *This is the same % error threshold that Module 03 (Agentic Metrics) and Module 05-03 (Strands) use — so you can compare scores across frameworks.*

During optimization (`trace is not None`), the metric returns a boolean so the optimizer knows pass/fail.

In [ ]:
def city_metric(example, prediction, trace=None):
    """Score based on % error: within 5% → 1.0, within 10% → 0.5, else → 0.0."""
    expected = extract_number(example.answer)
    predicted = extract_number(prediction.answer)

    if expected is None or predicted is None or expected == 0:
        return False if trace else 0.0

    pct_error = abs(predicted - expected) / expected

    if pct_error <= 0.05:
        score = 1.0
    elif pct_error <= 0.10:
        score = 0.5
    else:
        score = 0.0

    if trace is not None:
        return score >= 0.5
    return score

# Quick test
score = city_metric(test_example, result)
print(f"Score: {score}")

## 8. How does the baseline score across all questions?

Run the metric across every test example. No more vibe-checking a few outputs by hand.

In [ ]:
evaluator = dspy.Evaluate(
    devset=testset,
    metric=city_metric,
    num_threads=1  # single-threaded to avoid Bedrock throttling in workshops,
    display_progress=True,
    display_table=5,
)

baseline_score = float(evaluator(baseline))
print(f"\nBaseline average score: {baseline_score:.2f}")

# Find errors, not scores — which questions did the model get wrong?
print("\n--- Failure Analysis ---")
failures = []
for ex in testset:
    pred = baseline(question=ex.question)
    score = city_metric(ex, pred)
    if score < 1.0:
        expected = extract_number(ex.answer)
        predicted = extract_number(pred.answer)
        pct_err = abs(predicted - expected) / expected * 100 if predicted and expected else float('inf')
        failures.append((ex.question, ex.answer, pred.answer, pct_err, score))

failures.sort(key=lambda x: x[3], reverse=True)
for q, exp, pred, pct, score in failures[:5]:
    print(f"\n  Q: {q}")
    print(f"  Expected: {exp} | Got: {pred} | Error: {pct:.1f}% | Score: {score}")

print(f"\n{len(failures)} of {len(testset)} questions scored below perfect.")

### Wait — is that score stable?

Run the same evaluation again. Same model, same data, same metric.


In [ ]:
# Same evaluation, second run
baseline_score_2 = float(evaluator(baseline))
print(f"Run 1: {baseline_score:.2f}")
print(f"Run 2: {baseline_score_2:.2f}")
print(f"Difference: {abs(baseline_score - baseline_score_2):.2f}")


Your score might be 65 or 75 — the exact number matters less than the **pattern**. Scroll through the failures above. Are the same *types* of questions failing? Density questions are harder than population questions. That pattern is stable even when the scores aren't.

> **Find the errors, not the score.** That's the principle for the rest of this workshop.


## 8b. What if the metric itself is broken?

Before you optimize, make sure your metric actually measures what you think. Here's a common failure: the model gives a correct answer in a format your metric can't parse.

In [ ]:
# Simulate metric parsing on different response formats
test_responses = [
    "The population is approximately 780,995 people.",  # verbose but correct
    "780995",                                            # clean
    "About 780K",                                        # abbreviated
    "Seven hundred eighty thousand, nine hundred ninety-five",  # words
]

expected = 780995
print(f"Expected: {expected}\n")
for resp in test_responses:
    extracted = extract_number(resp)
    if extracted and expected > 0:
        pct_err = abs(extracted - expected) / expected
        score = 1.0 if pct_err <= 0.05 else (0.5 if pct_err <= 0.10 else 0.0)
    else:
        score = 0.0
    print(f"  '{resp}'")
    print(f"    Extracted: {extracted} \u2192 Score: {score}")
    print()

print("\u26a0\ufe0f  If your metric returns 0 for correct answers, optimization will learn the wrong thing.")
print("Always test your metric on edge cases before running the optimizer.")

## 9. Can the machine write a better prompt?

You want better prompts without writing them by hand. DSPy can find the best examples automatically — it runs the model on training examples, scores each output with your metric, and keeps the winners as few-shot demonstrations.

The optimizer that does this is called `BootstrapFewShot`.

⏱️ *This step makes ~40 API calls and takes 2–3 minutes. Estimated cost: ~$0.05 with Claude Haiku.*

In [ ]:
import time

def retry_on_throttle(fn, max_retries=3, base_delay=5):
    """Retry with exponential backoff on Bedrock throttling."""
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except Exception as e:
            if "ThrottlingException" in str(e) and attempt < max_retries:
                delay = base_delay * (2 ** attempt)
                print(f"\u23f3 Throttled — retrying in {delay}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(delay)
            else:
                raise

In [ ]:
optimizer = dspy.BootstrapFewShot(
    metric=city_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=4,
    max_rounds=2,
)

optimized = retry_on_throttle(lambda: optimizer.compile(baseline, trainset=trainset))

optimized_score = float(evaluator(optimized))

print(f"\n{'='*40}")
print(f"Baseline score:  {baseline_score:.2f}")
print(f"Optimized score: {optimized_score:.2f}")
print(f"Improvement:     {optimized_score - baseline_score:+.2f}")
print(f"{'='*40}")

### What if optimization makes things worse?

It happens. Common causes:
- **Bad metric** — the optimizer maximizes whatever you measure. If your metric is broken (see §8b), the optimizer will exploit it.
- **Too few examples** — with only 5–10 training examples, the optimizer may overfit to those specific questions.
- **Unrepresentative demos** — the bootstrapped examples may not generalize to your test set.

If your optimized score drops below baseline, go back to your metric and your training data before blaming the optimizer.

### Trust, but verify

The score jumped. But should you trust it? What did the optimizer actually change? Before we move on, let's crack open the optimized program and see exactly what it did — no black boxes.

## 10. What did the optimizer actually change?

The optimizer selected specific question/answer pairs as few-shot demonstrations. Let's see which ones it picked and what the full prompt looks like now.

In [ ]:
# What demos did the optimizer select?
print(f"Number of demos: {len(optimized.demos)}")
print()
for i, demo in enumerate(optimized.demos):
    print(f"--- Demo {i+1} ---")
    print(f"  Q: {demo.get('question', 'N/A')}")
    print(f"  A: {demo.get('answer', 'N/A')}")
    print()

# Run one prediction to see the full few-shot prompt
_ = optimized(question=testset[0].question)
dspy.inspect_history(n=1)

## 11. Can someone else use what you built?

Your colleague didn't attend this workshop. Can they use what you built? Yes — optimized programs are portable JSON artifacts. Save, commit to git, deploy to production, roll back if quality drops.

In [ ]:
import json as _json

# Save
optimized.save("optimized_city_qa.json")
print("Saved optimized program")

# Peek at the saved artifact
with open("optimized_city_qa.json") as f:
    saved = _json.load(f)
print(f"\nSaved artifact keys: {list(saved.keys()) if isinstance(saved, dict) else type(saved)}")
print(f"File size: {len(_json.dumps(saved))} bytes")

# Load into a fresh predictor
loaded = dspy.Predict(CityQA)
loaded.load("optimized_city_qa.json")

# Verify round-trip
loaded_score = float(evaluator(loaded))
print(f"\nOriginal optimized score: {optimized_score:.2f}")
print(f"Loaded program score:    {loaded_score:.2f}")
print(f"Round-trip OK: {abs(loaded_score - optimized_score) < 0.01}")

---
# Part 3: Go Deeper

You've measured, optimized, inspected, and saved. But can you make the model reason? Can you compose multiple steps? Can you catch errors that numbers alone miss?

## 12. What if the model showed its work?

`ChainOfThought` adds a `reasoning` field — the model thinks step-by-step before answering. One-line change from `Predict`.

In [ ]:
cot = dspy.ChainOfThought(CityQA)

# Show reasoning on one example
result = cot(question=testset[0].question)
print(f"Question:  {testset[0].question}")
print(f"Reasoning: {result.reasoning}")
print(f"Answer:    {result.answer}")
print(f"Expected:  {testset[0].answer}")

# Evaluate
cot_score = float(evaluator(cot))
print(f"\nChainOfThought score: {cot_score:.2f}")
print(f"vs Baseline:          {baseline_score:.2f}")
print(f"vs Optimized Predict: {optimized_score:.2f}")

## 13. How do you chain multiple steps? ⏱️ *Stretch*

When you need to chain multiple steps — or just want a reusable building block — you wrap them in a **Module**. Think of it as a recipe: define the ingredients in `__init__`, define the steps in `forward()`. The same optimizer that improved `Predict` works on modules too.

⏱️ *Another ~40 API calls for optimization. Estimated cost: ~$0.05.*

In [ ]:
class CityExpert(dspy.Module):
    def __init__(self):
        super().__init__()
        self.answer = dspy.ChainOfThought(CityQA)

    def forward(self, question):
        return self.answer(question=question)

module = CityExpert()
module_score = float(evaluator(module))
print(f"Module (CoT) score: {module_score:.2f}")

# Optimize the module
module_optimizer = dspy.BootstrapFewShot(
    metric=city_metric,
    max_bootstrapped_demos=3,
    max_labeled_demos=3,
)
optimized_module = retry_on_throttle(lambda: module_optimizer.compile(module, trainset=trainset))
optimized_module_score = float(evaluator(optimized_module))

print(f"\nModule (CoT) score:           {module_score:.2f}")
print(f"Module (CoT) + optimized:     {optimized_module_score:.2f}")
print(f"Improvement:                  {optimized_module_score - module_score:+.2f}")

## 14. What if numbers alone miss the problem? ⏱️ *Stretch*

Your numeric metric catches wrong numbers but misses made-up facts. A model could say *"Seattle's population is 780,995, located in Oregon"* — the number is right, but the context is hallucinated. You add a second check: ask an LLM to verify faithfulness.

💡 *This is the same LLM-as-Judge concept from Module 02 (Quality Metrics) — but here the judge lives inside the metric, and the metric drives optimization.*

⏱️ *The LLM-as-judge evaluation makes ~20 additional calls (one judge call per test example). Takes ~2 minutes.*
*Estimated cost: ~$0.03.*


In [ ]:
class FaithfulnessCheck(dspy.Signature):
    """Check if the answer is factually faithful and not hallucinated."""
    question: str = dspy.InputField()
    answer: str = dspy.InputField()
    expected_answer: str = dspy.InputField()
    is_faithful: bool = dspy.OutputField(desc="True if the answer is factually consistent with the expected answer")
    reason: str = dspy.OutputField(desc="Brief explanation of the judgment")

faithfulness_judge = dspy.Predict(FaithfulnessCheck)

def enhanced_metric(example, prediction, trace=None):
    """Numeric accuracy (70%) + LLM faithfulness judge (30%)."""
    expected = extract_number(example.answer)
    predicted = extract_number(prediction.answer)

    if expected is None or predicted is None or expected == 0:
        numeric_score = 0.0
    else:
        pct_error = abs(predicted - expected) / expected
        if pct_error <= 0.05:
            numeric_score = 1.0
        elif pct_error <= 0.10:
            numeric_score = 0.5
        else:
            numeric_score = 0.0

    # LLM judge component
    try:
        judgment = faithfulness_judge(
            question=example.question,
            answer=prediction.answer,
            expected_answer=example.answer
        )
        judge_score = 1.0 if judgment.is_faithful else 0.0
    except Exception:
        judge_score = 0.0

    score = 0.7 * numeric_score + 0.3 * judge_score

    if trace is not None:
        return score >= 0.5
    return score

# Re-evaluate the optimized module with the enhanced metric
enhanced_evaluator = dspy.Evaluate(
    devset=testset,
    metric=enhanced_metric,
    num_threads=1  # single-threaded to avoid Bedrock throttling in workshops,
    display_progress=True,
    display_table=5,
)

enhanced_score = float(enhanced_evaluator(optimized_module))
print(f"\nOptimized module with numeric metric:   {optimized_module_score:.2f}")
print(f"Optimized module with enhanced metric:  {enhanced_score:.2f}")

# Where do the two metrics disagree?
print("\n--- Where the judge disagrees with the number ---")
disagree_count = 0
for ex in testset[:10]:
    pred = optimized_module(question=ex.question)
    num = city_metric(ex, pred)
    enh = enhanced_metric(ex, pred)
    if abs(num - enh) > 0.1:
        disagree_count += 1
        print(f"\n  Q: {ex.question}")
        print(f"  Answer: {pred.answer}")
        print(f"  Numeric metric: {num:.2f} | Enhanced metric: {enh:.2f}")
        if disagree_count >= 3:
            break
if disagree_count == 0:
    print("  No disagreements in first 10 examples — try running again (non-deterministic).")


---
# The Full Picture

## 15. Which approach actually won?

Let's put every approach side by side — but remember, highest score isn't the only dimension that matters.

| Dimension | Predict | + Optimized | CoT | Module + Optimized | Enhanced Metric |
|-----------|---------|-------------|-----|--------------------|-----------------|
| **Accuracy** | Lowest | Higher | Mid | Highest | *Different lens* |
| **Latency** | Fastest | Same | ~2x | ~2x | ~3x (judge calls) |
| **Cost/call** | Lowest | Same | ~2x | ~2x | ~3x |
| **Debuggability** | Low | Low | High | High | Highest (judge reasons) |
| **Portability** | N/A | JSON artifact | N/A | JSON artifact | N/A |

Pick the approach that matches your requirements — not just the one with the highest number.


In [ ]:
# Check which scores are available
_scores = {}
for name, var in [
    ("baseline_score", "baseline_score"),
    ("optimized_score", "optimized_score"),
    ("cot_score", "cot_score"),
    ("module_score", "module_score"),
    ("optimized_module_score", "optimized_module_score"),
    ("enhanced_score", "enhanced_score"),
]:
    val = globals().get(var)
    if val is not None:
        _scores[name] = val

_expected = {"baseline_score", "optimized_score", "cot_score", "module_score", "optimized_module_score"}
_missing = _expected - set(_scores.keys())
if _missing:
    print(f"⚠️  Missing scores (skipped sections): {', '.join(sorted(_missing))}")
    print("The comparison below only includes completed sections.\n")

results = [
    ("Predict (baseline)", _scores.get("baseline_score")),
    ("Predict + BootstrapFewShot", _scores.get("optimized_score")),
    ("ChainOfThought", _scores.get("cot_score")),
    ("Module (CoT)", _scores.get("module_score")),
    ("Module (CoT) + BootstrapFewShot", _scores.get("optimized_module_score")),
    ("Enhanced metric (numeric + judge)", _scores.get("enhanced_score")),
]
results = [(name, score) for name, score in results if score is not None]

if not results:
    print("No scores available. Run at least §8 (Baseline Evaluation) first.")
else:
    ref = results[0][1]
    print(f"{'Approach':<40} {'Score':>8} {'vs Baseline':>12}")
    print("─" * 62)
    for name, score in results:
        delta = score - ref
        bar = "█" * int(score / 2)
        print(f"{name:<40} {score:>8.2f} {delta:>+11.2f}  {bar}")

    best_name, best_score = max(results, key=lambda x: x[1])
    print(f"\n🏆 Best: {best_name} ({best_score:.2f})")
    print("\n💡 Note: the enhanced metric scores lower because the LLM judge is stricter.")
    print("   A lower score doesn't mean worse — it means the metric catches more problems.")


## Conclusion

In this module you:

1. **Declared** a city Q&A task as a typed DSPy signature
2. **Measured** quality with a percentage-error metric (5% / 10% thresholds)
3. **Optimized** prompts automatically with `BootstrapFewShot`
4. **Inspected** what the optimizer changed — no black boxes
5. **Added reasoning** with `ChainOfThought`
6. **Composed** a custom module and optimized it end-to-end
7. **Enhanced** the metric with an LLM-as-judge faithfulness check

### Key Takeaway

> **Define what good looks like, measure it, and let the machine write the prompt for you.**

### FAQ

- **What happens when I update my model version?** Re-run the evaluation to check for regressions. Re-optimize if scores drop — the saved JSON is tied to the model that generated the demos.
- **Can I use this in production?** Yes — save the optimized program (§11), deploy it as a versioned artifact, and gate deployments on eval score thresholds.
- **How does this compare to Bedrock Prompt Management?** Bedrock Prompt Management versions your prompts manually. DSPy generates optimized prompts automatically from your metric. They're complementary — optimize with DSPy, then store the result in Prompt Management.

### Next Steps

- Try `MIPROv2` for instruction optimization
- Explore newer optimizers like `GEPA` and `SIMBA`
- Build multi-stage pipelines (retrieve → reason → answer)
- Save optimized programs to S3 for deployment
- Add evaluation gates to CI/CD pipelines

### Resources

- [DSPy Documentation](https://dspy.ai/)
- [DSPy Optimizers Guide](https://dspy.ai/learn/optimization/optimizers/)
- [Amazon Bedrock Documentation](https://docs.aws.amazon.com/bedrock/)